### Embeddings

In [1]:
#! pip install spacy
#! python -m spacy download en_core_web_md
#! python -m spacy download pt_core_news_sm
import spacy

In [2]:
# https://spacy.io/models/pt
nlp = spacy.load("en_core_web_md")
dog_emb = nlp.vocab["dog"].vector

In [3]:
print(type(dog_emb))
print(dog_emb.shape)
print(dog_emb[0:10])

<class 'numpy.ndarray'>
(300,)
[-0.72483   0.42538   0.025489 -0.39807   0.037463 -0.29811  -0.28279
  0.29333   0.57775   1.2205  ]


#### Chromadb

In [10]:
#! pip install chromadb
#! pip install -U sentence-transformers

In [6]:
import chromadb
from chromadb.utils import embedding_functions

In [7]:
CHROMA_DATA_PATH = "chroma_data/"
EMBED_MODEL = "all-MiniLM-L6-v2"
COLLECTION_NAME = "demo_docs"
client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)

In [11]:
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL)

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Carolina\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carolina\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o

In [12]:
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"},)

In [13]:
documents = [
    "The latest iPhone model comes with impressive features and a powerful camera.",
    "Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.",
    "Einstein's theory of relativity revolutionized our understanding of space and time.",
    "Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.",
    "The American Revolution had a profound impact on the birth of the United States as a nation.",
    "Regular exercise and a balanced diet are essential for maintaining good physical health.",
    "Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history.",
    "Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
    "Startup companies often face challenges in securing funding and scaling their operations.",
    "Beethoven's Symphony No. 9 is celebrated for its powerful choral finale, 'Ode to Joy.'",
]

In [14]:
genres = [
    "technology",
    "travel",
    "science",
    "food",
    "history",
    "fitness",
    "art",
    "climate change",
    "business",
    "music"
    ]

In [15]:
collection.add(
    documents=documents,
    ids=[f"id{i}" for i in range(len(documents))],
    metadatas=[{"genre": g} for g in genres]
    )

In [16]:
query_results = collection.query(
    query_texts=["Find me some delicious food!"],
    n_results=1)

In [17]:
query_results["documents"]

[['Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.']]

In [ ]:
query_results = collection.query(
    query_texts=["Teach me about history",
                  "What's going on in the world?"],
    include=["documents", "distances"],
    n_results=2
)

In [ ]:
query_results["documents"][0]
[
  "Einstein's theory of relativity revolutionized our understanding of space and time.",
  "The American Revolution had a profound impact on the birth of the United States as a nation."
]

In [ ]:
query_results["distances"][0]

In [ ]:
query_results["documents"][1]
[
  "Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
  "Einstein's theory of relativity revolutionized our understanding of space and time."
]

In [ ]:
query_results["distances"][1]